# Module 6: Multi-agent systems in production

1. Version 3 of the bot: a router and two specialist agents.
2. **Exercise 10 (15 min):** multi-agent autopsy.
3. Agent metrics: task success and trajectory efficiency.
4. A production log: latency, rates by topic, what to sample for review, escalations.
5. Evals in CI: a suite threshold versus per-check gates.

Exercise 11 (your production eval plan) is on paper.

In [ ]:
# Setup: run this cell first. It works in Google Colab and on your own laptop.
import os, sys
REPO_URL = "https://github.com/MarinaWyss/evaluating-ai-systems"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EvalsWorkshop"):
        !git clone -q {REPO_URL} /content/EvalsWorkshop
        !pip install -q "litellm>=1.80.5" tenacity
    os.chdir("/content/EvalsWorkshop")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from beefcake import llm
llm.load_colab_secrets()
if llm.has_api_key():
    print("API key found. Bot model:", llm.get_model())
else:
    print("No API key found, so this notebook runs in offline mode with pre-generated data.")

## 1. Version 3: a router, a handoff, and two agents

In [ ]:
from beefcake.agents import answer_v3, load_agent_traces, show_agent_trace

v3 = {t.trace_id: t for t in load_agent_traces("v3_traces.jsonl")}
if llm.has_api_key():
    show_agent_trace(answer_v3("can I still return the bell from order BC-4417? it's too heavy for me"))
else:
    show_agent_trace(v3["M2"])

## 2. Exercise 10: Multi-agent autopsy (15 min)

This trace has a wrong final answer.

1. Read it span by span and write open codes.
2. Which agent and which span went wrong first?
3. Check your codes against MAST's categories: system design, inter-agent misalignment, task verification.
4. Write the eval that would have caught it: a code check, a handoff assertion, or a judge.

In [ ]:
show_agent_trace(v3["M1"], full=True)

In [ ]:
my_autopsy = {
    "open codes": [],
    "agent and span that went wrong first": "",
    "MAST category": "",
    "eval that would catch it": "",
}

A handoff is recorded like a tool call (`transfer_to_...`), so you can assert on it. Write a routing check:

In [ ]:
from beefcake.checks import routed_correctly

def expected_agent(query):
    # YOUR CODE HERE: return "orders_billing" or "device_support"
    return "device_support"

for tid, t in v3.items():
    print(tid, t.user_query, "->", "PASS" if routed_correctly(t, expected_agent(t.user_query)) else "FAIL")

With a key, test the live router on a small golden set:

In [ ]:
from beefcake.agents import route

router_golden = pd.read_csv("data/router_golden.csv")
if llm.has_api_key():
    router_golden["Routed to"] = [route(q) for q in router_golden["User Query"]]
    router_golden["Result"] = ["PASS" if a == b else "FAIL"
                               for a, b in zip(router_golden["Routed to"], router_golden["Expected agent"])]
router_golden

## 3. Agent metrics

Task success: did it reach the goal? Check the end state with code. Trajectory efficiency: the shortest path
that would have worked, divided by the path the agent took.

In [ ]:
import json
from beefcake import checks

tool_traces = load_agent_traces("exercise7_tool_calls.jsonl")
expected = pd.read_csv("data/exercise7_expected.csv").set_index("Trace ID")
SHORTEST = {"C09": 2}  # look up the order, then hand off. Everything else needs one call at most.

rows = []
for t in tool_traces:
    e = expected.loc[t.trace_id]
    rows.append({"Trace ID": t.trace_id, "User Query": t.user_query,
                 "Task success": checks.end_state_matches(t, json.loads(e["Expected returns"]), int(e["Expected tickets"]))
                                 and checks.no_false_success(t) and checks.arguments_match_schema(t)
                                 and checks.only_approved_tools(t),
                 "Tool calls": len(t.tool_calls),
                 "Efficiency": checks.trajectory_efficiency(t, SHORTEST.get(t.trace_id, 1))})
agent_metrics = pd.DataFrame(rows)
print(f"Task success: {agent_metrics['Task success'].mean():.0%}")
agent_metrics

The end state alone isn't enough for read-only requests. C02 changes nothing in the store, so its end state is
"right", but it told the customer their order doesn't exist. That's why task success here also requires valid
arguments and no made-up tools.

In [ ]:
# The least efficient runs
agent_metrics.sort_values("Efficiency").head(3)

## 4. Production

This log is **synthetic** (made up for the demo), with one row per conversation.

In [ ]:
log = pd.read_csv("data/production_log_synthetic.csv", keep_default_na=False)
lat = log["Latency (s)"]
print(f"{len(log)} conversations.  Latency p50: {lat.quantile(0.5):.1f}s   p99: {lat.quantile(0.99):.1f}s   mean: {lat.mean():.1f}s")
log.groupby("Query Topic").agg(
    conversations=("Trace ID", "count"),
    judge_fail_rate=("Judge: Assumes device", lambda s: (s[s != ""] == "FAIL").mean() if (s != "").any() else None),
    thumbs_down_rate=("Thumbs", lambda s: (s == "down").mean()),
    p99_latency=("Latency (s)", lambda s: s.quantile(0.99)),
).round(2)

### What should a person read this week?

Judge-flagged traces, negative feedback, outliers, and always some random ones.

In [ ]:
review = pd.concat([
    log[log["Judge: Assumes device"] == "FAIL"].sample(8, random_state=1).assign(Why="judge flagged"),
    log[log["Thumbs"] == "down"].sample(6, random_state=1).assign(Why="thumbs down"),
    log.nlargest(3, "Latency (s)").assign(Why="slowest"),
    log.sample(8, random_state=2).assign(Why="random"),
]).drop_duplicates("Trace ID")
print(len(review), "traces to read. (The Assumes-device judge only runs on device questions.)")
review[["Trace ID", "Query Topic", "Why"]]

### Evaluate the handoff to a person like a classifier

Use the conversations a person labeled.

In [ ]:
labeled = log[log["Should escalate (human label)"] != ""].copy()
labeled["should"] = labeled["Should escalate (human label)"].astype(str) == "True"
labeled["did"] = labeled["Escalated"].astype(str) == "True"
missed = int((labeled["should"] & ~labeled["did"]).sum())
unneeded = int((~labeled["should"] & labeled["did"]).sum())
print(f"Labeled conversations: {len(labeled)}")
print(f"Missed escalations:    {missed} of {int(labeled['should'].sum())} that needed a person")
print(f"Unneeded escalations:  {unneeded} of {int((~labeled['should']).sum())} that didn't")

## 5. Evals in CI

`scripts/run_ci_evals.py` runs code checks on every change. Compare the suite threshold with the per-check gates.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("run_ci_evals", "scripts/run_ci_evals.py")
ci = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ci)
report = ci.run()
print("Suite pass rate:", report["suite pass rate"], "->", report["suite gate"])
display(pd.DataFrame(report["per-check gates"]).T)
print("Ship it?", report["ship it"])

The suite clears its 90% threshold while three checks fail. A whole-suite threshold can hide a test that fails
every time, so anything important gets its own gate.

To explore traces span by span in Arize Phoenix, see `scripts/phoenix_demo.py`.